# Tables as Python Objects: A First Look at Pony ORM

## Can a program use the rows of a table as ordinary objects?

In [SQL from Python](https://dvirross.com/courses/intro-data-science/sql-from-python), the program sent SQL as text and got tuples back. An **[ORM](https://dvirross.com/courses/intro-data-science/glossary#orm)** (object-relational mapper) changes the division of work: you describe a table as a Python **class**, each row becomes an **object** of that class, and the ORM writes the SQL for you. This lesson uses **Pony ORM**, a Python library. It starts from the bike-share days as before, and then connects two tables that point to each other, with data from the World Bank.

Pony has a special trick that connects to [objects and generator expressions](https://dvirross.com/courses/intro-data-science/python-objects-and-generators): you write a query as a **[generator expression](https://dvirross.com/courses/intro-data-science/glossary#generator-expression)**, such as:

```python
d for d in Day if d.season == 'Summer'
```

Pony reads that expression and translates it into SQL. That is why that lesson spent time on the form of the expression.

**How to work:** run the cells from top to bottom, because the cells share their variables. A cell that says `# Your turn` is empty or holds the set-up of the task: write your code there, run it, and run the cell under it, which checks your answer without showing the solution. The notebook with solutions has the solution, the common mistakes and the expected output under every task. A database object and its entities are defined once in each run: if you run the cell that defines `db` or `Day` a second time, Pony stops with an error, so restart the session (in Colab: **Runtime**, then **Restart session**) and run the cells again from the top.

## Setup: install Pony and get the data file

No account and no key are needed. The next cell installs Pony, the library of this lesson, if the session does not have it yet (Colab does not include it, and the installation takes a few seconds). The cell after it downloads the small CSV file of the lesson from the public site and saves it next to the notebook (in Colab it appears in the **Files** panel). If the download fails, open the address below in a browser, download the file, and upload it to the **Files** panel of Colab.

`https://dvirross.com/materials/courses/intro-data-science/world-bank-more-countries.csv`

In [1]:
try:
    import pony
except ImportError:
    !pip install -q pony
    import pony
print('Pony is ready, version', pony.__version__)

Pony is ready, version 0.7.20


In [2]:
import os
import urllib.request

FILE_URL = 'https://dvirross.com/materials/courses/intro-data-science/world-bank-more-countries.csv'
FILE_PATH = 'world-bank-more-countries.csv'

if not os.path.exists(FILE_PATH):
    # the site refuses the default user agent of Python, so the request names one
    request = urllib.request.Request(FILE_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request) as response, open(FILE_PATH, 'wb') as file:
        file.write(response.read())
print('Data file ready:', FILE_PATH)

Data file ready: world-bank-more-countries.csv


The next cell defines the function `check(task_number)`. After you run the cell of a task, run the cell under it: `check` runs your cell again, collects what it prints, and compares it with the expected output. It tells you whether the output is right, and where it differs if not; it never shows the solution.

In [3]:
import ast, contextlib, hashlib, io

# The expected output of each task, as hashes of its lines (the solutions are not stored here).
EXPECTED = {int(k): v for k, v in {
    '1': [
        '2dd29fb6f518',
        'e1ae88365d47',
        '396a1bf61e9a'
    ],
    '2': [
        '6f42ae6ab479',
        '5c558c73b770'
    ],
    '3': [
        '2dd29fb6f518',
        'e1ae88365d47',
        '396a1bf61e9a'
    ],
    '4': [
        'f5ca38f748a1',
        'e7f6c011776e'
    ],
    '5': [
        'dc289ab8c8fa',
        '7b2c7298ef49',
        '401cb6939b81',
        '7413e6942dbb',
        '6cceb9bc987f',
        '5e0fdfdc44f9'
    ],
    '6': [
        '8ee58a0b6923',
        '770859d608eb'
    ],
    '7': [
        'ee7e860a857f'
    ],
    '8': [
        '809c025ba41f',
        'dc937b598926'
    ],
    '9': [
        '1d564088a318',
        'e18cad10fc80'
    ],
    '10': [
        '2b2bc2ccad4c',
        '51b2762f9f61'
    ],
    '11': [
        '59e1e5985094'
    ]
}.items()}


def check(task):
    """Run the last cell that ran (not a check) again and compare what it prints with the expected output of the task."""
    if task not in EXPECTED:
        print('There is no such task: ', task)
        return
    cells = [s for s in In[:-1] if s.strip() and not s.lstrip().startswith('check(')]   # In: the cells that ran, kept by IPython
    tree = ast.parse(cells[-1] if cells else '')
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None   # a bare last expression counts as printed
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(compile(tree, '<task>', 'exec'), globals())
            if last is not None:
                value = eval(compile(ast.Expression(last.value), '<task>', 'eval'), globals())
                if value is not None:
                    print(repr(value))
    except Exception as error:
        print('The cell raised an error:', type(error).__name__ + ':', error)
        return
    got = [hashlib.sha256(line.rstrip().encode()).hexdigest()[:12] for line in out.getvalue().rstrip().splitlines()]
    want = EXPECTED[task]
    if not got:
        print('Nothing was printed. Print the answer in the task cell, run it, and run `check` straight after it.')
    elif len(got) != len(want):
        print('The cell printed {got} line(s) and the task expects {want}. Re-read the task and the hint.'.format(got=len(got), want=len(want)))
    elif got != want:
        print('Line {k} is different from the expected output. Re-read the task and the hint.'.format(k=next(i for i in range(len(want)) if got[i] != want[i]) + 1))
    else:
        print('OK: the output is as expected.')


## Describe a table as a class

A class that stands for a table is called an **[entity](https://dvirross.com/courses/intro-data-science/glossary#entity)**. It inherits from `db.Entity`, where `db` is the object that represents the database, and each column is a line that names the attribute and its kind: `Required(str)` is a text that must have a value, and `Required(int)` is a whole number that must have a value. The first line of the next box, `from pony.orm import *`, brings in every name that Pony offers. Three names of Pony do the set-up. `Database()` creates the object `db` that represents a database. `db.bind(...)` connects it to SQLite; `':memory:'` is again a temporary database in memory, as in [SQL from Python](https://dvirross.com/courses/intro-data-science/sql-from-python). `db.generate_mapping(...)` finishes the description of the tables; with `create_tables=True` it also creates the table in the database, if it does not exist yet.

In [4]:
from pony.orm import *

db = Database()
db.bind(provider='sqlite', filename=':memory:')

class Day(db.Entity):
    date = Required(str)
    season = Required(str)
    working_day = Required(int)
    rentals = Required(int)

db.generate_mapping(create_tables=True)
print(list(db.entities))

['Day']


Reading the lines in order:

- `from pony.orm import *` brings in the names of Pony. The star means "every name that the module offers", as in [values and collections](https://dvirross.com/courses/intro-data-science/python-values-and-collections). It includes `sum`, `min` and `max`, which keep working on ordinary lists and numbers as before.
- The class `Day` describes the table `Day`: one row for each day, with the columns `date`, `season`, `working_day` and `rentals`. Pony also adds a column `id` that numbers the rows. It is the **[primary key](https://dvirross.com/courses/intro-data-science/glossary#primary-key)**, so you did not need to write it.
- `print(list(db.entities))` shows the names of the entities that the database knows about.

## Add rows as objects

In a program, Pony talks to the database only inside a `with db_session:` block (see [db_session](https://dvirross.com/courses/intro-data-science/glossary#db-session)). When the block ends normally, Pony saves the changes; if an error stops the block, Pony cancels them. Creating a row is creating an object: call the class with the values of the columns.

In [5]:
rows = [
    ('2011-01-01', 'Winter', 0, 985),
    ('2011-01-02', 'Winter', 0, 801),
    ('2011-01-03', 'Winter', 1, 1349),
    ('2011-07-04', 'Summer', 0, 6043),
    ('2011-07-05', 'Summer', 1, 4665),
    ('2011-07-06', 'Summer', 1, 4629),
]

with db_session:
    for date, season, working_day, rentals in rows:
        Day(
            date=date, season=season, working_day=working_day,
            rentals=rentals
        )
    print(Day.select().count())
    day = Day[1]
    print(day, day.date, day.rentals)

6
Day[1] 2011-01-01 985


The loop uses the [unpacking](https://dvirross.com/courses/intro-data-science/glossary#unpacking) from [loops and functions](https://dvirross.com/courses/intro-data-science/python-loops-and-functions), and the six rows are the real days from the SQL lessons. `Day[1]` fetches the object whose primary key is 1, and its attributes are read with a dot, as in [objects and generator expressions](https://dvirross.com/courses/intro-data-science/python-objects-and-generators). Printing the object shows the class name and the key. Use the objects inside the `with db_session:` block, where Pony is connected to the database.

### What happens outside a db_session block

Every use of entities belongs inside `with db_session:` in a program that Python runs from a file: a query outside the block stops with the error `TransactionError: db_session is required when working with the database`. A notebook behaves differently, as the explanation below says, so the cells of this notebook keep the block in every query, as the page does.

**Pony in a notebook (Jupyter or Colab)**

Pony decides how strictly to require `db_session` when it is first imported. A program that Python runs from a file is not interactive, and Pony requires the block there. A notebook (Jupyter or Colab) is an **interactive** console: Pony notices that and lets a query run outside `with db_session:`. The same line `print(Day.select().count())` prints the number of rows there instead of stopping with an error.

Keep writing `with db_session:` in a notebook anyway. The code then works unchanged when it moves into a program, and the end of the block is where Pony saves the changes, or cancels them after an error. In Colab, Pony is installed with `pip install pony` in a cell, because Colab does not include it.

## Read rows: a query as a generator expression

`select(...)` takes a generator expression. The part before `for` says what to return, the part after `in` says which entity, and the condition after `if` filters the rows, just like `SELECT`, `FROM` and `WHERE`. Looping over the result runs the query and gives one object for each row. The method `order_by` sorts the rows of a result by the value of the small function you give it, and `desc(...)` makes the order from the largest down, like `ORDER BY ... DESC`.

In [6]:
with db_session:
    summer = select(d for d in Day if d.season == 'Summer')
    for d in summer.order_by(lambda d: desc(d.rentals)):
        print(d.date, d.rentals)

2011-07-04 6043
2011-07-05 4665
2011-07-06 4629


To see the SQL that Pony wrote, the method `get_sql` of a query returns it as text:

In [7]:
with db_session:
    print(
        select(
            d for d in Day if d.season == 'Summer'
        ).get_sql()
    )

SELECT "d"."id", "d"."date", "d"."season", "d"."working_day", "d"."rentals"
FROM "Day" "d"
WHERE "d"."season" = 'Summer'


This is a query you could have typed in DB Browser. Pony also uses the **parameters** of [SQL from Python](https://dvirross.com/courses/intro-data-science/sql-from-python): a Python variable inside the expression is not pasted into the text, but sent as a value for a question mark.

In [8]:
with db_session:
    limit = 4600
    busy = select(d for d in Day if d.rentals > limit)
    print(busy.get_sql())
    print([d.date for d in busy])

SELECT "d"."id", "d"."date", "d"."season", "d"."working_day", "d"."rentals"
FROM "Day" "d"
WHERE "d"."rentals" > ?
['2011-07-04', '2011-07-05', '2011-07-06']


A **[slice](https://dvirross.com/courses/intro-data-science/glossary#slice)** takes the first rows only, as with a list: `select(...)[:2]` returns the first two objects.

When only one object is needed, the method `first()` is shorter than `list(...)[0]`. It returns the first object of a result, and `None` when the result is empty, where `list(...)[0]` stops with an error on an empty list. It differs from the slice `[:1]` as well: the slice returns a list that holds at most one object, and `first()` returns the object itself. After `order_by(...)`, the first object is the first in that order. The box takes the day with the fewest rentals, and then asks for a day above 10000 rentals, which does not exist:

In [9]:
with db_session:
    quietest = select(d for d in Day).order_by(
        lambda d: d.rentals
    ).first()
    print(quietest.date, quietest.rentals)
    huge = select(d for d in Day if d.rentals > 10000).first()
    print(huge)

2011-01-02 801
None


The first call returns an object, whose attributes are read with a dot, and the second finds no day and returns `None`, so check for `None` before reading an attribute of the result.

### Practice

**Your turn.** Inside a `with db_session:` block, print the dates of the days of `'Winter'`, one on each line, from the earliest date to the latest.

In [10]:
# Your turn

**Expected output (the numbers are hidden)**

```text
<number>-<number>-<number>
<number>-<number>-<number>
<number>-<number>-<number>
```

In [11]:
check(1)

> **Hint**
>
> Keep the winter days with a condition after `if`, sort the result with `order_by(lambda d: d.date)`, and loop over it to print one attribute on each line.

## Summaries and groups

The summary functions of SQL have names in Pony too: `count`, `sum`, `avg`, `min` and `max`. Each takes a generator expression that gives the values to summarize.

In [12]:
with db_session:
    print(count(d for d in Day))
    print(avg(d.rentals for d in Day if d.season == 'Summer'))
    print(max(d.rentals for d in Day))
    print(sum(d.rentals for d in Day if d.working_day == 1))

6
5112.333333333333
6043
10643


When the expression returns a pair, such as the season and a summary, Pony groups by the part that is not summarized:

In [13]:
with db_session:
    by_season = select(
        (d.season, avg(d.rentals)) for d in Day
    )
    print(by_season.get_sql())
    print(list(by_season))

SELECT "d"."season", AVG("d"."rentals")
FROM "Day" "d"
GROUP BY "d"."season"
[('Summer', 5112.333333333333), ('Winter', 1045.0)]


The same shape counts the rows of each group: with `count(d)` in the pair, where `d` is the whole day, Pony counts the days of each season. It counts each day once, by its primary key, which is why the SQL has `COUNT(DISTINCT ...)`.

In [14]:
with db_session:
    days_per_season = select(
        (d.season, count(d)) for d in Day
    )
    print(days_per_season.get_sql())
    print(list(days_per_season))

SELECT "d"."season", COUNT(DISTINCT "d"."id")
FROM "Day" "d"
GROUP BY "d"."season"
[('Summer', 3), ('Winter', 3)]


The SQL has a `GROUP BY` that you did not write: the [SQL guide, Part 1](https://dvirross.com/courses/intro-data-science/sql-in-db-browser) taught it, and here the ORM adds it from the shape of the expression.

### The same total in two ways

A question about a total can be answered by a loop in Python or by a summary function, and both give the same number. The method `select()` of an entity, with no arguments, returns all its objects. The loop below takes every day from the database and adds the rentals of the summer days one by one, in Python. The summary function asks the database for the total instead:

In [15]:
with db_session:
    total = 0
    for d in Day.select():
        if d.season == 'Summer':
            total = total + d.rentals
    print(total)
    print(sum(d.rentals for d in Day if d.season == 'Summer'))

15337
15337


The loop brings all the days into Python, and `sum(...)` makes Pony send one `SELECT SUM(...)` and receive one number. For six days the two take equally long. For a table of many thousands of rows, the loop would still move every row into Python, and `sum(...)` would still receive one number.

### A summary that feeds a second query

A summary is an ordinary Python value, so it can be stored in a variable and used in the next query. The first query below finds the largest rentals with `max`, and `Day.get(rentals=top)` then returns the day with that value. The method `get` expects exactly one match: when two days share the value, Pony stops with an error, and `select(...).first()` is the way to take one of them. The second query finds the average rentals of all days with `avg`, and a query keeps the days above it. Pony sends the stored value as a parameter, as in the query with `limit` above.

In [16]:
with db_session:
    top = max(d.rentals for d in Day)
    busiest = Day.get(rentals=top)
    print(busiest.date, top)

    mean = avg(d.rentals for d in Day)
    above = select(d for d in Day if d.rentals > mean)
    print(round(mean), above.count())

2011-07-04 6043
3079 3


The average of the six days is about 3079, and three days have more rentals than that: the three summer days.

### Practice

**Your turn.** Inside a `with db_session:` block, print the largest rentals among the winter days, using `max`, and on the next line the average rentals of the winter days, using `avg`.

In [17]:
# Your turn

**Expected output (the numbers are hidden)**

```text
<number>
<number>.<number>
```

In [18]:
check(2)

> **Hint**
>
> Each summary function takes a generator expression, and both lines need the same condition after `if`: the season is `'Winter'`.

**Your turn.** Inside a `with db_session:` block, store the average rentals of all the days in a variable. Then print the dates of the days with fewer rentals than that average, one on each line, from the earliest date to the latest.

In [19]:
# Your turn

**Expected output (the numbers are hidden)**

```text
<number>-<number>-<number>
<number>-<number>-<number>
<number>-<number>-<number>
```

In [20]:
check(3)

> **Hint**
>
> Write the average in a variable inside the block, use that variable after `if` in a `select(...)`, and sort the result with `order_by(lambda d: d.date)`.

## Change and delete rows

To change a column, assign to the attribute of the object. To delete a row, call `delete()` on its object. We first add a **test day** that is not real data, so that the real days stay as they are. `Day.get(...)` returns the one object that matches, or `None` if there is none.

In [21]:
with db_session:
    Day(
        date='2099-01-01', season='Summer', working_day=1,
        rentals=1
    )

with db_session:
    test_day = Day.get(date='2099-01-01')
    test_day.rentals = 2
    print(test_day.rentals)

with db_session:
    print(Day.get(date='2099-01-01').rentals)
    Day.get(date='2099-01-01').delete()
    print(count(d for d in Day))
    print(Day.get(date='2099-01-01'))

2
2
6
None


The second block changed the value and the third block, a new session, still sees it: the change was saved when the second block ended. Compare this with [SQL from Python](https://dvirross.com/courses/intro-data-science/sql-from-python), where you wrote `UPDATE` and `DELETE` as SQL text and decided when to call `commit()`. Here the end of the `db_session` block is the commit, and an error inside the block is the rollback. You can also cancel on purpose: the call `rollback()` inside the block cancels every change made since the block began, as `conn.rollback()` does in [the section on changing data and saving it](https://dvirross.com/courses/intro-data-science/sql-from-python#changing-data-and-saving-it), and the block then continues with the database as it was.

The call `commit()` inside the block saves the changes made so far at once, and the block goes on. A later `rollback()` then cancels only what came after the `commit()`. The box below adds a test day, saves it with `commit()`, adds a second test day and cancels it with `rollback()`. A new block then reads what the database holds, and deletes the first test day, so that the table has its six real days again:

In [22]:
with db_session:
    Day(
        date='2097-01-01', season='Summer', working_day=1,
        rentals=1
    )
    commit()
    Day(
        date='2097-01-02', season='Summer', working_day=1,
        rentals=2
    )
    rollback()

with db_session:
    print(Day.get(date='2097-01-01').rentals)
    print(Day.get(date='2097-01-02'))
    Day.get(date='2097-01-01').delete()
    print(count(d for d in Day))

1
None
6


The first test day was saved by `commit()`, so it is still there, and the second was cancelled before it was saved, so `Day.get` finds nothing and returns `None`.

### Practice

**Your turn.** In one `with db_session:` block, create a test day with the date `'2098-01-01'`, the season `'Winter'`, `working_day` 0 and rentals 10. Change its rentals to 20 and print them. Then delete the day and print `count(d for d in Day)`.

In [23]:
# Your turn

**Expected output (the numbers are hidden)**

```text
20
<number>
```

In [24]:
check(4)

> **Hint**
>
> Keep the object that `Day(...)` returns in a variable. Assign to its attribute `rentals` to change it, and call `delete()` on the object to remove the row.

## A table that points to another table

In the [SQL guide](https://dvirross.com/courses/intro-data-science/sql-change-and-join), the department column of a table of employees holds the primary key of a row in another table, the departments. A column like that is a **[foreign key](https://dvirross.com/courses/intro-data-science/glossary#foreign-key)**, and a `JOIN` follows it to put each row next to the row it points to. When one row of the first table is pointed to by many rows of the second, as one country is pointed to by its yearly measurements, this is a **[one-to-many relation](https://dvirross.com/courses/intro-data-science/glossary#one-to-many-relation)**. An ORM writes the same link as attributes: an observation object holds its country object, and a country object holds all of its observation objects.

The data are real and come from the World Bank's **World Development Indicators** (license CC BY 4.0), retrieved on 30 October 2026 through the World Bank API. Two indicators are used. `SP.DYN.LE00.IN` is the **life expectancy at birth**: the number of years a newborn would live if the death rates of that year stayed the same for life. `NY.GDP.PCAP.PP.CD` is the **GDP per capita, PPP**: the value of what a country produces in a year per person, in current international dollars, a unit that corrects for differences in prices between countries. This is a small extract: 10 countries chosen to span regions and income levels, for the years 2000 and 2020. The income level of each country is the World Bank's own classification, as published at retrieval.

Three cautions come with the data. A few countries are not the world, so the extract can show a pattern but not measure it. Two measurements that move together are an association and not a proof that one causes the other. Also, a value that the source does not have stays missing: it is not zero. The GDP of Syria in 2000 is missing in the source, so it is missing here too.

We open a second database object, `geo`, because the `Day` tables were created in `db` already. A few names describe the relation and the new kinds of columns:

- `Set('Observation')` is the attribute `observations` of a country. It holds the set of all the observations that point to that country, and it is not a column of the table: Pony reads it from the column on the other side. The class name is in quotes because `Observation` is defined below it.
- `Required(Country)` is the attribute `country` of an observation. Its value must be one `Country` object, so every observation belongs to a country. `Optional(Country)` would allow an observation with no country.
- `Optional(float)` is a number with a decimal point that may be missing. A missing value is `None` in Python and `NULL` in SQL.

The function `show(Observation)` prints the description of an entity as Pony understood it: its attributes and their kinds, including the `id` that Pony added.

In [25]:
geo = Database()
geo.bind(provider='sqlite', filename=':memory:')

class Country(geo.Entity):
    code = Required(str)
    name = Required(str)
    income_level = Required(str)
    observations = Set('Observation')

class Observation(geo.Entity):
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)

geo.generate_mapping(create_tables=True)
show(Observation)

class Observation(Entity):
    id = PrimaryKey(int, auto=True)
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)


### Create the objects and connect them

The extract is two lists of tuples: the countries, and the observations, each with the code of its country, the year, the life expectancy and the GDP per capita, as the source publishes them.

In [26]:
country_rows = [
    ('ISR', 'Israel', 'High income'),
    ('JPN', 'Japan', 'High income'),
    ('USA', 'United States', 'High income'),
    ('NOR', 'Norway', 'High income'),
    ('BRA', 'Brazil', 'Upper middle income'),
    ('CHN', 'China', 'Upper middle income'),
    ('EGY', 'Egypt, Arab Rep.', 'Lower middle income'),
    ('IND', 'India', 'Lower middle income'),
    ('ETH', 'Ethiopia', 'Low income'),
    ('SYR', 'Syrian Arab Republic', 'Low income'),
]
print(len(country_rows))

10


In [27]:
observation_rows = [
    ('ISR', 2000, 78.9536585365854, 25746.8295829392),
    ('ISR', 2020, 82.6487804878049, 40969.2987342816),
    ('JPN', 2000, 81.0760975609756, 27694.5384332959),
    ('JPN', 2020, 84.56, 43560.021428549),
    ('USA', 2000, 76.6365853658537, 36329.9702595751),
    ('USA', 2020, 76.9804878048781, 64465.2971415748),
    ('NOR', 2000, 78.6341463414634, 36813.8379065645),
    ('NOR', 2020, 83.209756097561, 69779.2862630574),
    ('BRA', 2000, 69.584, 9091.76851342904),
    ('BRA', 2020, 74.506, 16101.6183063466),
    ('CHN', 2000, 72.288, 2963.43487212206),
    ('CHN', 2020, 78.019, 18267.4783455827),
    ('EGY', 2000, 67.332, 5451.51566974242),
    ('EGY', 2020, 69.79, 15231.865417764),
    ('IND', 2000, 62.749, 2030.76576304986),
    ('IND', 2020, 70.156, 6766.06119055994),
    ('ETH', 2000, 50.88, 473.903313465592),
    ('ETH', 2020, 65.969, 2406.78938480121),
    ('SYR', 2000, 70.856, None),
    ('SYR', 2020, 71.944, 3878.42802993832),
]
print(len(observation_rows))

20


The values of a dictionary can be any objects, not only numbers and text. We build a dictionary whose keys are the country codes and whose values are the `Country` objects. To create an observation of a country, we look the country object up by its code and pass it as `country=...`. That connects the two rows: Pony stores the key of the country in the column `country` of the observation. The value `None` in a row goes into an `Optional` column as a missing value.

In [28]:
with db_session:
    countries = {}
    for code, name, income in country_rows:
        countries[code] = Country(
            code=code, name=name, income_level=income
        )
    for code, year, life, gdp in observation_rows:
        Observation(
            year=year, life_expectancy=life,
            gdp_per_capita=gdp, country=countries[code]
        )
    print(
        Country.select().count(),
        Observation.select().count()
    )

10 20


### Follow the relation in both directions

Inside a `db_session`, a dot follows the relation. For an observation `obs`, `obs.country` is its `Country` object, and `obs.country.name` is the name of that country. In the other direction, `country.observations` is the set of the observations of a country. A set can be looped over like a list, and it has three methods: `count()` returns the number of objects in it, `order_by(...)` sorts it as for a query, and `select(...)` keeps the objects for which the function you give is true. To take one object by its place, as from a list, first turn the result of `order_by` or `select` into a list with `list(...)`.

In [29]:
with db_session:
    israel = Country.get(code='ISR')
    print(israel.name, israel.income_level)
    print(israel.observations.count())
    in_2020 = list(
        israel.observations.select(lambda o: o.year == 2020)
    )
    obs = in_2020[0]
    print(
        obs.country.name, obs.year,
        round(obs.life_expectancy, 1)
    )

Israel High income


2
Israel 2020 82.6


The function `round(x, 1)` rounds `x` to one digit after the decimal point. The first lines go from the country to its observations, and the last line goes back from an observation to its country.

The method `first()` of a query works on a set of related objects as well. The box takes the latest observation of Israel and then looks for a year that has no observation:

In [30]:
with db_session:
    israel = Country.get(code='ISR')
    latest = israel.observations.order_by(
        lambda o: desc(o.year)
    ).first()
    print(latest.year, round(latest.life_expectancy, 1))
    missing = israel.observations.select(
        lambda o: o.year == 1990
    ).first()
    print(missing)

2020 82.6
None


The first call returns an observation, and the second finds no observation of 1990 and returns `None`.

### Questions that the related objects answer

For each country, `country.observations.order_by(lambda o: o.year)` holds its two observations, the earlier year first, and `list(...)` turns it into a list, so the index `[0]` is 2000 and `[1]` is 2020. The change in life expectancy is the value of 2020 minus the value of 2000. We collect each change with the name of its country in a list of tuples, and `sorted(..., reverse=True)` puts the largest change first:

In [31]:
with db_session:
    gains = []
    for country in Country.select():
        years = list(
            country.observations.order_by(lambda o: o.year)
        )
        early = years[0].life_expectancy
        late = years[1].life_expectancy
        gains.append((round(late - early, 1), country.name))
    ranked = sorted(gains, reverse=True)
    print(ranked[:3])
    print(ranked[-1])

[(15.1, 'Ethiopia'), (7.4, 'India'), (5.7, 'China')]
(0.3, 'United States')


Life expectancy rose in every country of the extract, and rose most in Ethiopia, which started lowest. The extract does not say why. The two years are a snapshot, and the smallest change, 0.3 years in the United States, can come from the long trend, from the fact that 2020 is a year of the pandemic, or from both; two points cannot separate them.

For the GDP per capita, a value can be missing, and then there is nothing to divide. The test `x is None` is true when `x` is the missing value, so the loop keeps the countries without a value apart, and for the others it computes how many times the GDP grew, the value of 2020 divided by the value of 2000:

In [32]:
with db_session:
    multiples = []
    missing = []
    for country in Country.select():
        years = list(
            country.observations.order_by(lambda o: o.year)
        )
        early = years[0].gdp_per_capita
        late = years[1].gdp_per_capita
        if early is None or late is None:
            missing.append(country.name)
        else:
            multiples.append(
                (round(late / early, 1), country.name)
            )
    print(sorted(multiples, reverse=True)[:3])
    print('missing:', missing)

[(6.2, 'China'), (5.1, 'Ethiopia'), (3.3, 'India')]
missing: ['Syrian Arab Republic']


China and Ethiopia multiplied their GDP per capita by more than five, from low levels, so their multiples are large, while the gain per person of Ethiopia (about 1,900 dollars) is small next to the gain of Norway (about 33,000), which is not on the list. A multiple and a difference answer two different questions. Syria is not in the ranking because its value for 2000 is missing, not because it is zero.

### A condition across the relation

Inside a query, a dot can follow the relation as well: `o.country.income_level` is the income level of the country of the observation `o`. Such a query needs both tables, and Pony writes the join that the SQL guide wrote by hand. We group the observations of 2020 by income level and average the two indicators; `avg` skips the missing values:

In [33]:
with db_session:
    by_level = select(
        (
            o.country.income_level,
            avg(o.life_expectancy),
            avg(o.gdp_per_capita)
        )
        for o in Observation if o.year == 2020
    )
    print(by_level.get_sql())
    by_gdp = sorted(by_level, key=lambda row: row[2])
    for level, life, gdp in by_gdp:
        print(level, round(life, 1), round(gdp))

SELECT "country"."income_level", AVG("o"."life_expectancy"), AVG("o"."gdp_per_capita")
FROM "Observation" "o", "Country" "country"
WHERE "o"."year" = 2020
  AND "o"."country" = "country"."id"
GROUP BY "country"."income_level"
Low income 69.0 3143
Lower middle income 70.0 10999
Upper middle income 76.3 17185
High income 81.8 54693


The `WHERE` has the condition on the year and the line that connects the foreign key `"o"."country"` to the primary key `"country"."id"`, as `ON` does in a `JOIN`. Higher income levels go with a longer life expectancy in this extract, which is an association between group averages of two to four countries each. It does not show that income produces health, or the other way around.

### The SQL that actually runs

The method `get_sql` shows the text of one query. The function `set_sql_debug(True)` is wider: from that point Pony prints every SQL command at the moment it sends it to the database, with a few lines about the connection, until `set_sql_debug(False)` stops it. Here it shows what following a relation costs:

In [34]:
with db_session:
    set_sql_debug(True)
    norway = Country.get(code='NOR')
    print(norway.observations.count())
    set_sql_debug(False)

GET CONNECTION FROM THE LOCAL POOL
SWITCH TO AUTOCOMMIT MODE
SELECT "id", "code", "name", "income_level"
FROM "Country"
WHERE "code" = ?
LIMIT 2
['NOR']

SELECT COUNT(*)
FROM "Observation"
WHERE "country" = ?
[4]

2


### A database that already exists

Until now `':memory:'` gave a new, empty database each time. A database file that exists already, made by another program, is connected with its file name: `geo.bind(provider='sqlite', filename='wdi_extract.db')`. Without the argument `create_db=True`, Pony does not create the file, so a wrong file name is an error and not a new empty database. The call `generate_mapping()` without `create_tables=True` creates nothing: it checks that the tables of the file exist and have the columns that the classes describe, and it stops with an error if they do not match. The classes must therefore describe the tables as they are, with the same names.

The block below is shown and not run, because it needs the file. We made the file `wdi_extract.db` in advance, from the classes of this section with `create_db=True`:

```python
geo = Database()
geo.bind(provider='sqlite', filename='wdi_extract.db')

class Country(geo.Entity):
    code = Required(str)
    name = Required(str)
    income_level = Required(str)
    observations = Set('Observation')

class Observation(geo.Entity):
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)

geo.generate_mapping()

with db_session:
    israel = Country.get(code='ISR')
    print(israel.name, israel.observations.count())
```

```text
Israel 2
```

### Rows from a file

A table often arrives as a [CSV file](https://dvirross.com/courses/intro-data-science/python-values-and-collections#data-file-formats): a text file with one row on each line and the names of the columns on the first line. The file `world-bank-more-countries.csv` has the same two indicators for four more countries, for the years 2000 and 2020, with one line for each country and year. The source and the license are those of the extract above; the file was retrieved on 5 November 2026. The GDP per capita of Eritrea for 2020 is missing in the source, so that field in the file is empty.

Python reads such a file with the module `csv`. The function `open(name, newline='', encoding='utf-8')` opens a file, and a `with` block closes it when the block ends; the option `newline=''` is what the module `csv` asks for. The function `csv.DictReader(file)` goes over the lines after the first one, and gives each line as a [dictionary](https://dvirross.com/courses/intro-data-science/glossary#dictionary) whose keys are the names of the columns. Every value that comes from a file is text, even a number, and an empty field is the empty text `''`. The function `int(text)` makes a whole number from text, and `float(text)` makes a number with a decimal point.

In [35]:
import csv

with open(
    'world-bank-more-countries.csv', newline='', encoding='utf-8'
) as file:
    rows = list(csv.DictReader(file))

print(len(rows))
print(rows[0]['name'], rows[0]['year'] * 2)
print(rows[0]['name'], int(rows[0]['year']) + 1)
print(rows[7]['gdp_per_capita'] == '')

8
Chile 20002000
Chile 2001
True


The year is text, so repeating it twice joins two copies of the text, and only `int(...)` makes it a number that can be added to. A missing value must not become a number: an empty field becomes `None` with a conditional expression, `float(text) if text != '' else None`, which is `None` when the field is empty and the number otherwise.

Loading the lines into entities is the loop of [Create the objects and connect them](#create-the-objects-and-connect-them), with `row['code']` in place of the unpacked code. A `Country` is created only the first time that its code appears: the test `code not in countries` is true when the dictionary has no key `code` yet. The method `get` takes several conditions at once: `Observation.get(year=2020, country=some_country)` returns the one observation that matches all of them.

### Practice

**Your turn.** Create a database object `wb_db` in memory, as in this section. Then write two entities, in this order. `Economy` has the attributes `code` and `name`, each a text that must have a value, and `readings`, a set of `Reading` objects. `Reading` has `indicator` (a text that must have a value), `year` (a whole number that must have a value), `value` (a number with a decimal point that may be missing) and `economy` (an `Economy` object that must have a value). Then create the tables with `generate_mapping` and print `show(Reading)`.

In [36]:
# Your turn

**Expected output (the numbers are hidden)**

```text
class Reading(Entity):
    id = PrimaryKey(int, auto=True)
    indicator = Required(str)
    year = Required(int)
    value = Optional(float)
    economy = Required(Economy)
```

In [37]:
check(5)

> **Hint**
>
> Each class inherits from `wb_db.Entity`. `Reading` is named inside `Set(...)` in quotes, because it is defined below `Economy`.

**Your turn.** The classes `Economy` and `Reading` below are ready, `names` holds the name of each code, and `rows` holds real readings of two economies: South Africa (`ZAF`) and Syria (`SYR`), whose GDP for 2000 is missing in the source. In a `with db_session:` block, create a dictionary `economies` whose keys are the codes and whose values are `Economy` objects, then create a `Reading` for each row, connected to its economy through the dictionary. Then print, for each code in the dictionary, the code and the number of its readings, with `count()`.

In [38]:
wb_db = Database()
wb_db.bind(provider='sqlite', filename=':memory:')

class Economy(wb_db.Entity):
    code = Required(str)
    name = Required(str)
    readings = Set('Reading')

class Reading(wb_db.Entity):
    indicator = Required(str)
    year = Required(int)
    value = Optional(float)
    economy = Required(Economy)

wb_db.generate_mapping(create_tables=True)
names = {'ZAF': 'South Africa', 'SYR': 'Syrian Arab Republic'}
rows = [
    ('ZAF', 'life', 2000, 58.368),
    ('ZAF', 'life', 2020, 65.15),
    ('ZAF', 'gdp', 2020, 12671.0862436851),
    ('SYR', 'life', 2000, 70.856),
    ('SYR', 'life', 2020, 71.944),
    ('SYR', 'gdp', 2000, None),
    ('SYR', 'gdp', 2020, 3878.42802993832),
]

# Your turn

**Expected output (the numbers are hidden)**

```text
ZAF <number>
SYR <number>
```

In [39]:
check(6)

> **Hint**
>
> In a `with db_session:` block, first loop over `names` and put `Economy(code=..., name=...)` in the dictionary under each code. Then loop over `rows`, and pass `economy=economies[code]` to `Reading`. A last loop over `economies` prints the code and `economies[code].readings.count()`.

**Your turn.** The box below stores the economy with the code `'ZAF'` and three of its readings. In a new `with db_session:` block, get that economy with `Economy.get(code='ZAF')` and print the change in its life expectancy between 2000 and 2020, rounded to one digit after the decimal point: the value of 2020 minus the value of 2000, using the readings with the indicator `'life'`.

In [40]:
wb_db = Database()
wb_db.bind(provider='sqlite', filename=':memory:')

class Economy(wb_db.Entity):
    code = Required(str)
    name = Required(str)
    readings = Set('Reading')

class Reading(wb_db.Entity):
    indicator = Required(str)
    year = Required(int)
    value = Optional(float)
    economy = Required(Economy)

wb_db.generate_mapping(create_tables=True)

rows = [
    ('life', 2020, 65.15),
    ('life', 2000, 58.368),
    ('gdp', 2020, 12671.0862436851),
]

with db_session:
    zaf = Economy(code='ZAF', name='South Africa')
    for indicator, year, value in rows:
        Reading(
            indicator=indicator, year=year,
            value=value, economy=zaf
        )

# Your turn

**Expected output (the numbers are hidden)**

```text
<number>.<number>
```

In [41]:
check(7)

> **Hint**
>
> The set `readings` of the economy has the methods `select` and `order_by`. Keep the readings whose indicator is `'life'`, sort them by year, and subtract the value at index `[0]` from the value at index `[1]`.

**Your turn.** The classes `Country` and `Observation` on the database `file_db` below are ready, and the file `world-bank-more-countries.csv` is in the working folder. Read the file with `csv.DictReader`. In a `with db_session:` block, create a dictionary `countries` from each code to its `Country` object, creating each country once, and create an `Observation` for each line, with the year as a whole number and the two values as numbers, or `None` when the field is empty. Print the number of countries and the number of observations on one line, separated by a space, and on the next line the GDP per capita of Eritrea (`ERI`) in 2020.

In [42]:
file_db = Database()
file_db.bind(provider='sqlite', filename=':memory:')

class Country(file_db.Entity):
    code = Required(str)
    name = Required(str)
    income_level = Required(str)
    observations = Set('Observation')

class Observation(file_db.Entity):
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)

file_db.generate_mapping(create_tables=True)

# Your turn

**Expected output (the numbers are hidden)**

```text
<number> <number>
None
```

In [43]:
check(8)

> **Hint**
>
> Import `csv`, open the file as in the box above, and make a list of the lines with `list(csv.DictReader(file))`. In the block, loop over the lines: create the `Country` only when its code is not yet a key of `countries`, and always create the `Observation`, with `country=countries[code]`. The last line is `Observation.get(...)` with the country and the year, and then the attribute `gdp_per_capita`.

## Practice

Each question has its own code box: write your answer there and select **Run**. Open a hint if you are stuck, and the solution only after you try.

Before the questions, notice what stays the same from the SQL lessons: the tables, the keys and the thinking about which rows and which summary you need. Only the way you write the request changes.

**Question 1.** Print the date and rentals of the working days (`working_day` is 1) with more than 4600 rentals, from the busiest day down.

In [44]:
# Your turn

**Expected output (the numbers are hidden)**

```text
<number>-<number>-<number> <number>
<number>-<number>-<number> <number>
```

In [45]:
check(9)

> **Hint**
>
> Put both conditions after `if`, joined with `and`, and sort with the function:
>
> ```python
> order_by(lambda d: desc(d.rentals))
> ```
>
> Loop over the result and print two attributes.

**Question 2.** For each season, print how many days had more than 1000 rentals, one line for each season: the season, a space, the number.

In [46]:
# Your turn

**Expected output (the numbers are hidden)**

```text
Summer <number>
Winter <number>
```

In [47]:
check(10)

> **Hint**
>
> Return a pair, the season and `count(d)`, and put the condition on `d.rentals` after `if`.

**Question 3.** Add the real day `'2011-07-07'` (a summer working day with 4592 rentals), then print the average rentals of the summer days, rounded to a whole number. Finish with `rollback()`, which cancels the new day, so that the other boxes of the page still see the same six days.

In [48]:
# Your turn

**Expected output (the numbers are hidden)**

```text
<number>
```

In [49]:
check(11)

> **Hint**
>
> Create the object with `Day(...)` inside a `db_session` block, then call `avg` on a generator expression about the summer days. Wrap it in `round(...)`.

## Keep in mind

- An **ORM** maps a table to a class and a row to an object. **Pony** writes the SQL from a generator expression, so the form of the expression matters.
- `db.Entity` classes describe tables, `Required(...)` describes a column, and `db.generate_mapping(...)` creates the tables.
- Work with the database inside `with db_session:`: creating objects, running queries and reading related objects. A program that runs outside it stops with an error. The end of the block saves the changes, and an error inside cancels them.
- `select(...)`, `count`, `avg`, `sum`, `min` and `max` correspond to `SELECT`, `COUNT`, `AVG`, `SUM`, `MIN` and `MAX`; `get_sql()` shows the SQL that Pony wrote.
- To change a column, assign to the attribute; to delete a row, call `delete()`.
- `commit()` inside the block saves the changes so far; a later `rollback()` cancels only what came after it.
- A summary can be stored in a variable and used in a second query; `get(...)` returns the one object that matches and stops with an error when several do.
- A **one-to-many relation** is a foreign key seen from both sides: `Required(Country)` or `Optional(Country)` on the many side, `Set('Observation')` on the one side. A dot follows it in both directions, and inside a query a dot such as `o.country.income_level` makes Pony write the join.
- `Optional(float)` allows a missing value, which is `None`; test it with `is None` before calculating. A missing value is not zero.
- `first()` returns the first object of a query result, or `None` when it is empty.
- A dictionary from a code to an object (`countries[code]`) connects new objects to the right related object.
- `csv.DictReader` reads a CSV file: every value is text, so convert it with `int(...)` or `float(...)`, and turn an empty field into `None`.
- A notebook lets a query run outside `with db_session:` and a program does not; keep the block in both.
- `show(Entity)` prints how Pony understood an entity, and `set_sql_debug(True)` prints every SQL command that runs. To use a database file that exists, `bind` it without `create_db=True` and call `generate_mapping()` without `create_tables=True`.